# RNN vs LSTM vs GRU

In this lab, we will explore the three kings of Sequence Modeling:
1.  **Vanilla RNN**: The original, simple, but forgetful.
2.  **LSTM (Long Short-Term Memory)**: The complex veteran that remembers everything.
3.  **GRU (Gated Recurrent Unit)**: The efficient, younger sibling of LSTM.

We will train **all three models** on the exact same dataset to see how they compare in performance and implementation.

### Suggested Datasets for Real-World Practice
While we will use synthetic data (Sine Wave) today to ensure everyone's code runs perfectly, here are excellent real-world datasets you can try later by simply replacing the `y` variable:
1.  **International Airline Passengers**: A classic dataset showing an upward trend + seasonality.
2.  **Daily Minimum Temperatures**: Great for pure seasonality without much trend.
3.  **Stock Prices (e.g., AAPL, GOOGL)**: Highly chaotic and noisy (very hard to predict!).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import time # To time our training!

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

### 1. The Dataset (Shared Ground)
We will generate 1000 points of a Sine Wave. 
*   **Task**: Given the last 40 points, predict the next 1 point.

In [ ]:
# Generate 1000 points from 0 to 100
t = np.linspace(0, 100, 1000)

# specific relationship: y = sin(t) + some minor noise
y = np.sin(t) + np.random.normal(0, 0.05, 1000)

plt.figure(figsize=(12, 4))
plt.title("Synthetic Time Series Data")
plt.plot(y)
plt.grid(True)
plt.show()

In [ ]:
# Preprocessing

scaler = MinMaxScaler(feature_range=(-1, 1))
y_normalized = scaler.fit_transform(y.reshape(-1, 1))
print(f"Shape of y_normalized: {y_normalized.shape}")

In [ ]:
# Convert to Tensor - Flattening it to 1D array for simplicity

# This ensures train_y has shape (Batch_Size) not (Batch_Size, 1, 1)
y_tensor = torch.FloatTensor(y_normalized).view(-1) # view(-1) flattens the tensor to 1D

print(f"Shape of y_tensor: {y_tensor.shape}")

### Windowing Function or Preparing time-series or sequential data for recurrent neural networks (RNNs), including LSTMs and GRUs. 

- Primary function is to transform a univariate or multivariate time series into a dataset of input-output pairs suitable for supervised learning.

- When using create_sequences for training RNN models, it is important to consider the choice of window_size.
- A smaller window_size may capture short-term patterns but may not provide enough context for long-term dependencies, while a larger window_size may include too much historical data, leading to vanishing gradient problems in deep networks.

- For example, if we have a time series `[d1, d2, d3, d4, d5]` and a `window_size` of 3:
    - The first input sequence would be `[d1, d2, d3]`, with `d4` as its label.
    - The second input sequence would be `[d2, d3, d4]`, with `d5` as its label.
    - This process continues until the end of the input data is reached, ensuring that each generated sequence has a corresponding future data point as its target.


In [ ]:
def create_sequences(input_data, window_size):
    sequences = [] 
    labels = []

    L = len(input_data)

    for i in range(L - window_size):
        seq = input_data[i:i+window_size]
        label = input_data[i+window_size]

        sequences.append(seq) # Appending sequences
        labels.append(label)

    return torch.stack(sequences), torch.stack(labels)

In [ ]:
WINDOW_SIZE = 40 # Number of previous points to consider
X, z = create_sequences(y_tensor, WINDOW_SIZE) # Creating sequences and labels 

print(f"Sequence shape: {X.shape}")
print(f"Label shape: {z.shape}")

### Generated sequence

In [ ]:
plt.figure(figsize=(12, 4))
plt.title("Generated sequence X")
plt.plot(z)
plt.grid(True)
plt.show()

In [ ]:
# Train/Test Split

test_size = 100
train_X, train_y = X[:-test_size], z[:-test_size]
test_X, test_y = X[-test_size:], z[-test_size:]

print(f"Train Shape: {train_X.shape} (Batch, Sequence Length)")
print(f"Test Shape: {test_X.shape}")

### 2. The Contenders: Model Architectures

We will define three classes. 

Notice their similarity!

In [ ]:
# 1. The Vanilla RNN
class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, output_size=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True) # batch_first=True means input and output have shape (batch, seq, feature)
        self.linear = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # Reshape to (Batch, Sequence, Feature)
        x = x.view(len(x), -1, 1)
        out, hidden_state = self.rnn(x) # output, hidden state
        return self.linear(out[:, -1, :])

# 2. The LSTM
class SimpleLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        x = x.view(len(x), -1, 1) 
        out, hidden_state = self.lstm(x)
        return self.linear(out[:, -1, :])

# 3. The GRU
class SimpleGRU(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, output_size=1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        x = x.view(len(x), -1, 1)
        out, hidden_state = self.gru(x)
        return self.linear(out[:, -1, :])

### In the `forward` function, why did we do `rnn_out[:, -1, :]`?

### 3. Training Function
To be fair, we must treat all models equally. We use the same loop, same optimizer settings, and same epochs.

In [ ]:
def train_model(model, name):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    epochs = 50
    
    start_time = time.time()
    losses = []
    
    model.train()
    for i in range(epochs):
        optimizer.zero_grad()
        y_pred = model(train_X)
        
        # Shape Check:
        # y_pred will be [Batch, 1], train_y is [Batch]
        # We reshape y_pred to [Batch] to match train_y perfectly
        loss = criterion(y_pred.view(-1), train_y)
        
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        
    duration = time.time() - start_time
    print(f"{name} Training Completed in {duration:.2f} seconds. Final Loss: {losses[-1]:.5f}")
    return losses, duration

### 4. Instantiate all three models and train them.

In [ ]:
# Initialize
rnn_model = SimpleRNN()
lstm_model = SimpleLSTM()
gru_model = SimpleGRU()

# Train
rnn_losses, rnn_time = train_model(rnn_model, "Vanilla RNN")
lstm_losses, lstm_time = train_model(lstm_model, "LSTM")
gru_losses, gru_time = train_model(gru_model, "GRU")

### 5. Results & Discussion

Let's visualize the results perfectly. We will look at:
1.  **Training Speed (Convergence)**: Who learned faster?
2.  **Prediction Accuracy**: Who predicted the Future best?

In [ ]:
# Plot 1: Training Loss Comparison
plt.figure(figsize=(12, 5))
plt.plot(rnn_losses, label=f'RNN (Time: {rnn_time:.2f}s)')
plt.plot(lstm_losses, label=f'LSTM (Time: {lstm_time:.2f}s)')
plt.plot(gru_losses, label=f'GRU (Time: {gru_time:.2f}s)')
plt.title("Training Loss Reduction")
plt.xlabel("Epochs")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot 2: Final Predictions on Test Data
def get_predictions(model):
    model.eval()
    with torch.no_grad():
        pred = model(test_X).numpy()
    return scaler.inverse_transform(pred)

rnn_pred = get_predictions(rnn_model)
lstm_pred = get_predictions(lstm_model)
gru_pred = get_predictions(gru_model)
actual = scaler.inverse_transform(test_y.reshape(-1, 1))

plt.figure(figsize=(14, 6))
plt.plot(actual, label='Actual Data', color='black', linewidth=2)
plt.plot(rnn_pred, label='RNN Prediction', linestyle='--')
plt.plot(lstm_pred, label='LSTM Prediction', linestyle='--')
plt.plot(gru_pred, label='GRU Prediction', linestyle='--')

plt.title("Final Test Predictions")
plt.legend()
plt.grid(True)
plt.show()

### Analysis

1.  **Speed**: Did GRU train faster than LSTM? Or observe the loss graph.
2.  **Accuracy**: Did the Basic RNN or LSTM or GRU fail? 
3.  **Overfitting**: Look at the loss graph. Did any model "plateau" (stop learning) earlier than others?
4.  Tune the hyperparameters to improve the model.